In [1]:
import os, sys

PROJECT_ROOT = '/scratch/jq2uw/derm_vlms'
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from eval.metrics import eval_model, compute_metrics

VLMS = {
    'MedGemma':     'results/medgemma_predictions_all.csv',
    'GPT-5.3':      'results/gpt53_predictions_all.csv',
    'DermatoLlama': 'results/dermato_llama_predictions_all.csv',
}

dfs = {}
metrics = {}
for name, path in VLMS.items():
    full = os.path.join(PROJECT_ROOT, path)
    if not os.path.exists(full):
        print(f'[SKIP] {name}: {path} not found')
        continue
    df = pd.read_csv(full)
    df = eval_model(df)
    dfs[name] = df
    metrics[name] = compute_metrics(df)
    print(f'{name}: {len(df)} rows loaded')

print(f'\nLoaded {len(dfs)} models')

MedGemma: 1300 rows loaded
GPT-5.3: 1222 rows loaded
DermatoLlama: 2252 rows loaded

Loaded 3 models


In [2]:
# Table 1: Overall top-1 / top-3 accuracy
rows = []
for name, m in metrics.items():
    ov = m['overall'].iloc[0]
    rows.append({
        'Model': name,
        'N': int(ov['n']),
        'Parsed (%)': f"{ov['parsed'] / ov['n'] * 100:.1f}",
        'Matched (%)': f"{ov['matched'] / ov['n'] * 100:.1f}",
        'Top-1 Acc (%)': f"{ov['top1_acc'] * 100:.1f}",
        'Top-3 Acc (%)': f"{ov['top3_acc'] * 100:.1f}",
    })
table1 = pd.DataFrame(rows)
print('=== Overall Accuracy ===')
table1

=== Overall Accuracy ===


,Model,N,Parsed (%),Matched (%),Top-1 Acc (%),Top-3 Acc (%)
0,MedGemma,1300,100.0,100.0,29.9,52.1
1,GPT-5.3,1222,97.8,97.6,28.6,50.5
2,DermatoLlama,2252,67.2,64.0,18.5,32.4


In [3]:
# Table 2: Accuracy by image_mode
frames = []
for name, m in metrics.items():
    t = m['by_image_mode'].copy()
    t.insert(0, 'Model', name)
    frames.append(t)
table2 = pd.concat(frames, ignore_index=True)
table2['top1_acc'] = (table2['top1_acc'] * 100).round(1)
table2['top3_acc'] = (table2['top3_acc'] * 100).round(1)
table2 = table2.rename(columns={'top1_acc': 'Top-1 (%)', 'top3_acc': 'Top-3 (%)', 'n': 'N'})
print('=== Accuracy by Image Mode ===')
table2

=== Accuracy by Image Mode ===


,Model,image_mode,Top-1 (%),Top-3 (%),N
0,MedGemma,photo,29.9,50.7,422
1,MedGemma,dscope,30.7,55.5,420
2,MedGemma,combined,32.2,55.3,416
3,MedGemma,virtual,0.0,0.0,42
4,GPT-5.3,photo,25.7,46.9,397
5,GPT-5.3,dscope,29.6,52.2,395
6,GPT-5.3,combined,33.5,57.5,391
7,GPT-5.3,virtual,0.0,0.0,39
8,DermatoLlama,photo,10.7,21.9,718
9,DermatoLlama,dscope,23.0,36.9,716


In [4]:
# Table 3: Accuracy by y3 class
frames = []
for name, m in metrics.items():
    t = m['by_y3'].copy()
    t.insert(0, 'Model', name)
    frames.append(t)
table3 = pd.concat(frames, ignore_index=True)
table3['top1_acc'] = (table3['top1_acc'] * 100).round(1)
table3['top3_acc'] = (table3['top3_acc'] * 100).round(1)
table3 = table3.rename(columns={'top1_acc': 'Top-1 (%)', 'top3_acc': 'Top-3 (%)', 'n': 'N', 'ground_truth': 'y3'})
print('=== Accuracy by y3 Class ===')
table3

=== Accuracy by y3 Class ===


,Model,y3,Top-1 (%),Top-3 (%),N
0,MedGemma,malignant,44.0,71.2,580
1,MedGemma,benign,25.3,50.0,526
2,MedGemma,other,0.5,0.5,194
3,GPT-5.3,malignant,28.8,58.1,539
4,GPT-5.3,benign,38.6,59.8,502
5,GPT-5.3,other,0.6,2.2,181
6,DermatoLlama,malignant,17.5,31.8,935
7,DermatoLlama,benign,27.3,46.4,926
8,DermatoLlama,other,0.0,0.8,391


In [5]:
# Table 4: Accuracy by y16 class
frames = []
for name, m in metrics.items():
    t = m['by_y16'].copy()
    t.insert(0, 'Model', name)
    frames.append(t)
table4 = pd.concat(frames, ignore_index=True)
table4['top1_acc'] = (table4['top1_acc'] * 100).round(1)
table4['top3_acc'] = (table4['top3_acc'] * 100).round(1)
table4 = table4.rename(columns={'top1_acc': 'Top-1 (%)', 'top3_acc': 'Top-3 (%)', 'n': 'N'})
print('=== Accuracy by y16 Diagnosis ===')
table4

=== Accuracy by y16 Diagnosis ===


,Model,y16,Top-1 (%),Top-3 (%),N
0,MedGemma,Basal Cell Carcinoma,83.3,92.5,228
1,MedGemma,Melanocytic Nevus,49.8,74.7,225
2,MedGemma,Other Benign,7.3,23.0,178
3,MedGemma,Actinic Keratosis,17.5,56.7,97
4,MedGemma,Seborrheic Keratosis,5.2,52.1,96
5,MedGemma,Squamous Cell Carcinoma,16.1,90.3,93
6,MedGemma,Squamous Cell Carcinoma In Situ,0.0,0.0,78
7,MedGemma,Melanoma,44.0,84.0,75
8,MedGemma,Melanocytic Lesion,0.0,0.0,46
9,MedGemma,Dermatofibroma,0.0,4.8,21


In [6]:
# Table 5: Unmatched diagnosis terms (for refining synonym dictionary)
for name, m in metrics.items():
    um = m['unmatched']
    print(f'\n=== {name}: {len(um)} unmatched terms ===')
    if not um.empty:
        print(um['unmatched_term'].value_counts().head(20))


=== MedGemma: 12 unmatched terms ===
unmatched_term
Infection                    2
Herpes Labialis              1
Vasculitis                   1
Insect Bites                 1
Drug Eruption                1
Miliaria                     1
Foreign Body Granuloma       1
Vitiligo                     1
Sebaceous Gland Carcinoma    1
Acanthosis Nigricans         1
Herpes Zoster                1
Name: count, dtype: int64

=== GPT-5.3: 107 unmatched terms ===
unmatched_term
Café‑au‑lait macule                                                                                                                                                                                                                                                                                                                      2
Café‑au‑lait macule (if clinically larger and uniformly pigmented).\n\nImage quality limits assessment of key dermoscopic criteria (network, globules, streaks, regression structures), so diagnosti

In [7]:
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
table1.to_csv(os.path.join(RESULTS_DIR, 'eval_overall.csv'), index=False)
table2.to_csv(os.path.join(RESULTS_DIR, 'eval_by_image_mode.csv'), index=False)
table3.to_csv(os.path.join(RESULTS_DIR, 'eval_by_y3.csv'), index=False)
table4.to_csv(os.path.join(RESULTS_DIR, 'eval_by_y16.csv'), index=False)
print('Saved to results/eval_*.csv')

Saved to results/eval_*.csv
